## WELCOME TO NUDIMAX

Richard Baker,¹ Lynn J. Bonomo,² Paola Guzmán² ³, Terry Gosliner²

¹Center for Comparative Genomics, California Academy of Sciences

²Department of Invertebrate Zoology & Geology ([Gosliner Slug Lab](https://sluglab.wordpress.com/))

³University of Puerto Rico at Cayey

(click to show/hide documentation)

#### OVERVIEW
This Python notebook is designed to facilitate data-wrangling for partitioned maximum-likelihood phylogenetic analysis projects that follow a typical workflow of the Gosliner Slug Lab at the California Academy of Sciences (hence, NUDIMAX = NUDIbranch MAXimum-likelihood*). This is all packaged as an interactive Python notebook for use in [Google Colab](https://colab.research.google.com) (and includes a basic user interface using their Forms feature, but relies on an Internet connection) or a [Jupyter notebook](https://jupyter.org/) (which requires interacting directly with the code, but can be installed and run locally). It is also portable to an HPC cluster, if one is available.

> **\*Note:** Since v0.16, NUDIMAX supports both maximum-likelihood and Bayesian inference.

#### FEATURES
This is designed to:
- Receive a CSV matrix of accession numbers and GenBank IDs as input (see example Table 1)
- Process that CSV and output a list of GenBank accession numbers for submission to [BatchEntrez](https://www.ncbi.nlm.nih.gov/sites/batchentrez).
- Process the GenBank download and produce separate FASTA files for each gene
- Remotely query the NCBI BLAST database to verify that GenBank accessions were assigned correctly*
- Align those sequences (using MAFFT), returning both FASTA and NEXUS alignment files for partitioned maximum-likelihood analysis (using IQ-TREE)
- Combine the alignment files into a partitioned NEXUS for use in Bayesian analysis (using MrBayes)
- Run both alignment packages
- Rename Newick branches (e.g., adding species names)

> **\*Note:** While this feature is included, a remote BLAST query is much slower than the BLAST web interface or using a local database.

#### METHODOLOGY
This is accomplished via a number of custom functions, which primarily use Biopython and Pandas. Importantly, these custom functions only wrangle and convert data; NUDIMAX does not perform any analysis or computation directly. Instead, it includes user-friendly wrappers that allow users to call peer-reviewed packages (including MAFFT, IQ-TREE, MrBayes, and optionally NCBI BLAST) without requiring command-line interaction, root access, or macOS/Windows/Linux compatibility issues.

#### DISCLAIMER
This notebook is provided "as-is" with no warranty. While this tool is designed to automate and streamline a standard workflow, it remains the user's responsibility to validate all data entering and exiting the program. Users are **strongly** encouraged to read the code and comments to ensure that this pipeline meets the specific needs of their project.

NUDIMAX is still in active development, and should be considered to be in an alpha state. While it has been tested with a number of inputs, it may contain bugs as yet unseen (and therefore, unsquashed).

#### CITATION
Although NUDIMAX was written to handle the Gosliner Lab's specific use case, it should be helpful for anyone using a similar workflow. If you use NUDIMAX in your research, please cite the project as shown below.

**Important:** Since NUDIMAX includes wrappers for other utilities (including [MAFFT](https://mafft.cbrc.jp/alignment/software/), [NCBI BLAST](https://blast.ncbi.nlm.nih.gov/doc/blast-help/references.html#references), [IQ-TREE](https://iqtree.github.io/), and [MrBayes](https://nbisweden.github.io/MrBayes/)), you must also cite the utilities you use.

**APA format:**

Baker, R., Bonomo, L., Guzmán, P., & Gosliner, T. (2026). NUDIMAX: An accessible data-wrangling pipeline for partitioned phylogenetic workflows (Version 0.15a). Retrieved from https://github.com/RichardMSBS/NUDIMAX


**BibTeX format:**

```
bibtex
@software{NUDIMAX},
  author = {Baker, Richard and Bonomo, Lynn and Guzmán, Paola and Gosliner, Terry},
  title = {NUDIMAX: An accessible data-wrangling pipeline and wrapper for partitioned phylogenetic workflows},
  version = {0.17a},
  year = {2026},
  url = {https://github.com/RichardMSBS/NUDIMAX}
}
```

#### FAQ, readme, changelogs
note: collapse to hide

In [ ]:
# Notes
# when using the Colab forms UX, strings do not need to be bounded in quotes.
#   if working with code directly, syntax becomes much more important:
#
# - Python input strings must be bounded in quotes: matched sets of ', ", or '''
#   e.g., voucher_matrix_to_genbank('Nudis.csv')     will work
#         voucher_matrix_to_genbank("Nudis.csv")     will also work
#         voucher_matrix_to_genbank('''Nudis.csv''') will work—but at what cost?
#         voucher_matrix_to_genbank(Nudis.csv)       will crash
#         voucher_matrix_to_genbank('Nudis.csv)      will also crash
#
# - similarly, input commands need opening and closing parenthesis
#     e.g., table_cleaner('Kim_2024_T2 - subset.csv')     will work
#           table_cleaner('Kim_2024_T2 - subset.csv'      will crash
#           table_cleaner(Kim_2024_T2 - subset.csv)       will crash
#           table_cleaner('Kim_2024_T2 - subset.csv)      will crash

In [ ]:
## changelogs
#  Features to include by Tuesday:
#    [ ] query GenBank automatically
#    [ ] flag suspiciously long/short sequences
#    [ ] test Forms UX for portability to HPC

### 0.17-alpha
##   general
#    [ ] add alerts for non-IUPAC characters
#    [ ] add dynamic terminal padding
#    [ ] add OR statements to improve handling default parameters in Colab forms

### 0.16.2 patch
##   export_to_drive()
#    [X] added user-friendly way to export scratch directory to Google Drive

### 0.16.1 patch
##   MrBayes_wrapper()
#    [X] added "resume" function to check for partial runs (e.g., server timeout)
#    [X] added ability to upload pre-defined block files instead of generating them

### 0.16-alpha
##   general
#    [X] improved Google Colab forms UX
#    [X] improved readme/documentation, including reference info
#    MrBayes_converter()
#    [X] added functionality converting Biopython .nex blocks to MrBayes format
#    [X] translated matrix now saved to new folder (v. overwrite)
#    [X] switched to file-streaming (.readline) vs line-by-line to save memory
#    MrBayes_wrapper()
#    [X] added MrBayes functionality

### 0.15-alpha (not standalone; rolled into 0.16)
##   general
#    MAFFT_wrapper()
#    [X] add nexus conversion/concatenation (for MrBayes)
#    [X] added ability to toggle reverse complement detection on/off
#    MAFFT_single()
#    [X] improved reverse-complement handling/corrections
#    sanitize_fasta_headers()
#    [X] collapsed input FASTA header renaming into a single shared function
#    voucher_matrix_to_genbank()
#    [X] collapsed input FASTA header renaming into sanitize_fasta_headers()
#    leaf_renamer()
#    [X] collapsed input FASTA header renaming into sanitize_fasta_headers()

### 0.14-alpha
##   general
#    [X] phase out condacolab (curl/wget) w/fixed versions
#    [X] add limited UX using Colab Forms
#    [X] switch to an input folder of aligned fastas (.afa) instead of a concatenated nexus
#    [X] correct and update documentation
##   voucher_matrix_to_genbank():
#    [X] added voucher/accession string sanitization (illegal character removal)
#    [X] improved efficiency by sanitizing in place and writing with .stack()
#    [X] automated FASTA concatenation
##   fasta_writer()
#    [X] removed; functionality incorporated into join_single_gene_FASTA()
#    rename_fastas()
#    [X] removed; functionality incorporated into join_single_gene_FASTA()

### 0.13-alpha
#    [X] make a list of Genbank descriptions to verify sequences
#    [X] check for packages before installing them (to save time)
#    [X] include BLAST to identify sequences needing reassignment
#    [X] align using MAFFT for automatic RC detection

### 0.12-alpha
#    [X] combine genbank_reader() and genbank_to_fasta() into one function
#    [X] expand renaming support
#    [X] rename several functions and variables

### 0.11-alpha
#    [X] combine legacy pipelines

## Run this chunk to setup NUDIMAX
note: In Google Colab, collapse this section for one-click setup

###### install dependencies

In [12]:
def setup():
  print(f'NUDIMAX: installing dependencies...')
  import importlib.util
  import subprocess

  # check for Biopython; install if req'd
  if importlib.util.find_spec('Bio') is None:
    print(f' ? biopython not found. attempting installation...')
    subprocess.run(['pip', 'install', 'biopython'])
  else:
    print(f'  ✓ biopython installed.')

  import Bio          # main biopython package
  from   Bio          import AlignIO # for handling alignments
  from   Bio          import Entrez  # for querying GenBank
  from   Bio.Nexus    import Nexus   # nexus-specific commands
  from   Bio          import Phylo   # for handling trees
  from   Bio          import SeqIO   # for handling sequences
  import datetime
  from   google.colab import drive   # for backups
  import io           # for handling text streams
  import os           # for interacting with directories, filenames, etc
  import pandas as pd # for wrangling data in tabular format
  import pathlib      # interact with file paths and extensions
  from   pathlib    import Path    # for filepath-specific commands
  import re           # REGEX - TLDR: a more powerful find-and-replace utility
  import shutil       # for compressing files (.zip)
  from   shutil       import copytree, ignore_patterns # for backups
  import subprocess   # for running BASH commands in Python
  from   subprocess import Popen   # for running BASH from Python
  from   subprocess import PIPE    # for running BASH from Python
  from   subprocess import CalledProcessError # for running BASH from Python
  import zipfile      # for unzipping .zip archives

  # installs the following analysis packages:
  #   BLAST   2.17
  #   IQTree  3.1.3
  #   MAFFT   7.526
  #   MrBayes 3.2.7

  # detects operating environment and makes a /bin folder
  print(f'  ? attempting to detect environment...')
  home = str(pathlib.Path('.').resolve())
  if home == '/content': environ = 'COLAB'
  elif str(Path.home()).split('/')[1] == 'home': environ = 'LINUX'
  else: print('  X unexpected environment detected. crashing...')
  home = pathlib.Path(home)
  print(f'  ✓ environment detected: {environ}')
  bin  = pathlib.Path('.').resolve()/'bin'
  bin.mkdir(exist_ok=True)

  # installs the necessary packages
  pkgs = [['ncbi-blast-2.17.0+', 'https://ftp.ncbi.nlm.nih.gov/blast/executables/blast+/LATEST/ncbi-blast-2.17.0+-x64-linux.tar.gz', home/'bin/ncbi-blast-2.17.0+/bin/blastn'],
          ['mafft-linux64',      'https://mafft.cbrc.jp/alignment/software/mafft-7.526-linux.tgz', home/'bin/mafft-linux64/mafft.bat'],
          ['mrbayes-3.2.7',      'https://github.com/NBISweden/MrBayes/releases/download/v3.2.7/mrbayes-3.2.7.tar.gz', home/'bin/mrbayes-3.2.7/bin/mb'],
          ['iqtree-3.1.3-Linux', 'https://github.com/iqtree/iqtree3/releases/download/v3.1.3/iqtree-3.1.3-Linux.tar.gz', home/'bin/iqtree-3.1.3-Linux/bin/iqtree3']]

  for pkg, url, exe in pkgs:
    if os.path.exists(exe): print(f'  ✓ {pkg} installed.')
    else:
      print(f'  ? {pkg} not found. attempting installation...')
      tarball_name = url.split('/')[-1]
      tarball_path = pathlib.Path(tarball_name)
      subprocess.run(['wget', url], check = True)
      subprocess.run(['tar', '-xzvf', tarball_name, '-C', bin], check = True)
      tarball_path.unlink()
      if pkg == "mrbayes-3.2.7": # MrBayes ships un-compiled
        mb_main = Path(bin / 'mrbayes-3.2.7').resolve()
        commands = [["./configure", f"--prefix={mb_main}"],
                  ["make"], ["make", "install"]]
        for command in commands:
          subprocess.run(command, cwd = mb_main, check = True, text=True)
      print(f'  ✓ {pkg} installed.')

setup()

NUDIMAX: installing dependencies...
  ✓ biopython installed.
  ? attempting to detect environment...
  ✓ environment detected: COLAB
  ✓ ncbi-blast-2.17.0+ installed.
  ✓ mafft-linux64 installed.
  ✓ mrbayes-3.2.7 installed.
  ✓ iqtree-3.1.3-Linux installed.


##### define functions

###### export_to_drive()

In [ ]:
def export_to_drive(destination, copy_all = False, source = ''):
  PAD = len('export_to_drive: ')*' '
  print(f'export_to_drive: verifying environment...')
  RunningInCOLAB = 'google.colab' in str(get_ipython())
  if RunningInCOLAB == False:
    print('export_to_drive: Google Colab environment not detected. Cannot back up to Google Drive.')
    return

  MyDrive = '/content/drive/MyDrive/'
  if destination == '':
    print('export_to_drive: no destination specified. generating default name.')
    now = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
    destination = f'NUDIMAX_backup_{now}'
  if source == '' and copy_all == False:
    print(f'export_to_drive: error: no file specified for backup.')
    print(f'{PAD}select a file to back up, or set copy_all = True.')
    return

  drive_home_path = '/content/drive'
  if os.path.isdir(drive_home_path) == False:
    print(f'{PAD}mounting Google Drive...')
    drive.mount(drive_home_path)
  else: print(f'{PAD}Google Drive detected.')

  # checks for the target dir and makes it if it doesn't exist
  destination = str(destination).strip('/')
  destination = re.sub('^content/drive/mydrive', '', destination, flags=re.IGNORECASE)
  destination = MyDrive + destination
  print(f'{PAD}backup destination set to: {destination}')
  Path(destination).mkdir(exist_ok=True, parents=True)

  sourcepath = Path(source)
  # universal backup: copy the whole Colab scratch directory to Drive
  if   copy_all == True:
    print(f'{PAD}backing up all files in Colab scratch drive...')
    shutil.copytree('.', destination, dirs_exist_ok=True,
                    ignore=ignore_patterns('drive', '.config', 'sample_data', 'bin'))

  # selective backup: check whether the input is a dir or a file
  elif sourcepath.is_file(): shutil.copy(sourcepath, destination,)
  elif sourcepath.is_dir():  shutil.copytree(sourcepath, destination, dirs_exist_ok=True)
  else: print(f'{PAD}Error: invalid source.')
  print(f'\n{PAD}backup operation completed. verify backup before closing NUDIMAX.')

###### sanitize_fasta_headers()

In [ ]:
# this function strips characters from FASTA headers that would break MAFFT or IQ-TREE.
# the user doesn't interact with this directly; it's called by other functions
def sanitize_fasta_headers(input_df_column):
  # removes MAFFT/IQ-TREE-illegal characters from VOUCHERS
  print(f'sanitize_fasta_headers: removing MAFFT/IQ-TREE-breaking characters...\n')
  clean_headers = input_df_column.astype(str).replace(r'[.\- ]+', '_', regex=True)
  clean_headers = clean_headers.replace(r'[^a-zA-Z0-9_]', '', regex=True)

  return clean_headers

###### voucher_matrix_to_genbank()

In [ ]:
# this section removes illegal characters from the input table
def voucher_matrix_to_genbank(voucher_matrix):
  input_basename = os.path.splitext(voucher_matrix)[0]
  #input_abspath  = os.path.join(input_filepath, input_filename)

  if os.path.splitext(voucher_matrix)[1].lower() == '.csv':
    print(f'voucher_matrix_to_genbank: {voucher_matrix}: reading...')

    # creates a new dataframe (copies the original and tracks changes)
    input_raw = pd.read_csv(voucher_matrix, index_col=False)
    all_clean = input_raw.copy() # not clean yet!

    # removes BatchEntrez-illegal characters from ACCESSIONS
    print(f'  removing BatchEntrez-illegal characters...')
    dirty_genbankIDs   = input_raw.iloc[:,1:]
    cleaned_genbankIDs = dirty_genbankIDs.replace(r'[^.a-zA-Z0-9\-_]', '', regex=True)

    # removes MAFFT/IQTree-illegal characters from VOUCHERS (FASTA headers)
    # this is also needed later (to rename Newick branches), so handled by a shared function
    dirty_headers = input_raw.iloc[:,:1]
    clean_headers = sanitize_fasta_headers(dirty_headers)

    # saves the genbankIDs and vouchers (FASTA headers) to the cleaned file
    all_clean.iloc[:,1:] = cleaned_genbankIDs
    all_clean.iloc[:,:1] = clean_headers

    # saves the output matrix
    print(f'  saving cleaned assignment matrix...')
    assignment_out_filename = f'{input_basename}_clean.csv'
    print(f'  {assignment_out_filename}: writing...')
    all_clean.to_csv(assignment_out_filename, index=False)

    # drops the header and input numbers
    print(f'  converting to .txt for BE submission...')
    entrez_trimmed  = all_clean.iloc[:, 1:]
    entrez_filename = f'{input_basename}_BE.txt'
    accession_list  = entrez_trimmed.stack().reset_index(drop=True)
    accession_list.to_csv(entrez_filename, index=False, sep = ' ', header=False)

    print(f'')
    print(f'  Genbank assignments: {assignment_out_filename}')
    print(f'  GenBank query file:  {entrez_filename}')

  elif os.path.splitext(voucher_matrix)[1].lower() != '.csv':
    print(f'{voucher_matrix} does not appear to be a CSV file. skipping...')

###### genbank_to_fastas(), join_single_gene_FASTA(), join_multi_gene_FASTAs(), genbank_concatenator()

In [ ]:
def genbank_to_fastas(file_prefix, assignment_matrix, genbank_input, genbank_tag = ''):
  print(f'genbank_to_fastas: initializing...')
  # loads the assignment matrix and gets a list of gene names
  if os.path.splitext(assignment_matrix)[1].lower() == '.csv':
    assignment_data = pd.read_csv(assignment_matrix, index_col=False)
    genes = assignment_data.columns.values[1:]
    print(f'  {len(assignment_data)} entries detected in {assignment_matrix}.\n')
    print(f'  genes detected: {genes}\n')
  elif os.path.splitext(assignment_matrix)[1].lower() != '.csv':
    print(f'  {assignment_matrix} does not appear to be a CSV file.')
    return # ends the function

  # duplicates the voucher column for renaming
  assignment_data['Voucher_new'] = assignment_data['Voucher']
  # defining the dict to build on

  keycounts       = {} # to handle duplicate vouchers (e.g., Kim 2024)
  for index, base_voucher in enumerate(assignment_data.iloc[:, 0]):
      if base_voucher not in keycounts:
       keycounts[base_voucher] = 1
       voucher_new = base_voucher
      elif base_voucher in keycounts:
        keycounts[base_voucher] += 1
        voucher_new = f'{base_voucher}_{keycounts[base_voucher]}'
        print(f'  {base_voucher} exists. Adding as {voucher_new}.')
      assignment_data.at[index, 'Voucher_new'] = voucher_new
  print(f'  {len(assignment_data)} unique (or unique-ified) entries in assignment matrix.\n')

  # we'll use the new voucher to quickly find assignments
  assignment_data.set_index('Voucher_new', inplace=True)
  # saving the old:new assignments for use later
  oldnew_basename = os.path.splitext(assignment_matrix)[0]
  assignment_data.to_csv(f'{oldnew_basename}_sample_names.csv')

  genbank_details = {}

  # verifies that the genbank file is a valid type
  genbank_extensions = ['.gb', '.gbk', '.genbank']
  if os.path.splitext(genbank_input)[1].lower() in genbank_extensions:
    # assigns output filename based on the input filename
    output_basename = os.path.splitext(genbank_input)[0]

    # opens the genbank file and saves it as a variable
    genbank_download = SeqIO.parse(genbank_input, "genbank")

    # makes a directory to put output files in
    #os.makedirs(output_foldername, exist_ok=True)
  else:
    print(f'  {genbank_input} does not appear to be a valid genbank file.')
    return # ends the function

  for record in genbank_download:
    genbankID   = record.name
    for gene in genes:
      # searches the assignment matrix for each accession name and returns the
      idx = assignment_data.index[assignment_data[gene] == genbankID]
      if len(idx) != 0: # if idx is blank, the accession name wasn't in that gene column
        for index_entry in idx:
          if index_entry not in genbank_details:
            genbank_details[index_entry] = {}
          genbank_details[index_entry][gene] = str(record.seq)
          genbank_details[index_entry][f'{gene}_desc'] = str(record.description)

  assignment_data.set_index(assignment_data.columns[0], inplace=True)

  output_foldername = f'{file_prefix}_out'
  os.makedirs(output_foldername, exist_ok=True)
  log_foldername    = f'{file_prefix}_logs'
  os.makedirs(log_foldername, exist_ok=True)

  for gene in genes:
    output_fasta_filename = f'{file_prefix}_{gene}.fasta'
    output_fasta_filepath = f'{output_foldername}/{output_fasta_filename}'
    output_log_filename   = f'{file_prefix}_{gene}.log'
    output_log_filepath   = f'{log_foldername}/{output_log_filename}'

    with open(output_fasta_filepath, 'w') as output:
      with open(output_log_filepath, 'w') as log:
        for voucher, voucher_entries in genbank_details.items():
          if gene in voucher_entries:
            output.write(f'>{voucher}\n{voucher_entries[gene]}\n')
            log.write(f'>{voucher}\n{genbank_details[voucher][f"{gene}_desc"]}\n\n')

    seq_count = sum(1 for record in SeqIO.parse(output_fasta_filepath, "fasta"))
    print(f'  {seq_count} {gene} entries written to {output_fasta_filename}.')

  shutil.make_archive(output_foldername, "zip", output_foldername)
  print(f'\n  fastas written to {output_foldername}.zip')
  shutil.make_archive(log_foldername, "zip", log_foldername)
  print(  f'  logs written to {log_foldername}.zip\n')
  return(genes, genbank_details)

def join_single_gene_FASTA(file_prefix, gene, input_archive_name, delim = '_', user_tag = '', genbank_details = None, folder = False):
  concat_fasta_filename = f'{file_prefix}_concat_{gene}.fasta'

  # if we're processing multiple inputs, make a concat directory
  if folder is not False:
    dirname = f'{file_prefix}_concat'
    os.makedirs(dirname, exist_ok=True)
    concat_fasta_filename = Path(dirname) / concat_fasta_filename

  print(f'join_single_gene_FASTA: initializing...')
  # makes the concat file for writing...
  with open(concat_fasta_filename, 'w') as output:
    print(f'  {concat_fasta_filename}: preparing for writing...')
    keycounts = {} # to handle duplicate IDs

    # if there are user-generated files
    if genbank_details is not None:
      for voucher, voucher_genes in genbank_details.items():
        keycounts[voucher] = 1
        if gene in voucher_genes:
          output.write(f'>{voucher}\n{voucher_genes[gene]}\n')

    # gets a list of file names in the archive
    with zipfile.ZipFile(input_archive_name, 'r') as zip_ref:
      file_names = zip_ref.namelist()

      # if a file is a FASTA, proceed
      extensions = ['.fasta', '.fa', '.fas', '.fsa', '.fna', '.ffn', '.faa', '.frn', '.seq']
      for fasta in file_names:
        fasta_zippath = Path(fasta)
        filename = os.path.basename(fasta)
        fasta_basename = os.path.splitext(filename)[0]
        fasta_geneID   = fasta_basename.split(delim)[-1]
        # skips non-FASTA files
        if os.path.splitext(fasta)[1].lower() not in extensions:
          print(f'  {fasta} does not appear to be a FASTA file. skipping...')
          pass # skips this file but continues to the next one

        # if the file is a FASTA and is for the correct gene...
        elif fasta_geneID == gene:
          print(f'  {fasta_zippath}: processing...')
          # loads the fasta file without having to unzip the archive
          with zip_ref.open(fasta) as input_handle:
            # converts the zipped file to text
            text_stream = io.TextIOWrapper(input_handle, encoding='utf-8')
            # parses the text with Biopython's SeqIO module
            temp_fasta = SeqIO.parse(text_stream, "fasta")
            for record in temp_fasta:
              base_voucher = re.sub(r'[.\- ]+', '_', record.description)
              base_voucher = re.sub(r'[^a-zA-Z0-9_]', '', base_voucher)
              base_voucher = f'{base_voucher}{user_tag}'
              record.description = ''
              if base_voucher not in keycounts:
                keycounts[base_voucher] = 1
                voucher_new = base_voucher
              elif base_voucher in keycounts:
                keycounts[base_voucher] += 1
                voucher_new = f'{base_voucher}_{keycounts[base_voucher]}'
                print(f'  {base_voucher} exists. Adding as {voucher_new}.')
              record.id = voucher_new
              SeqIO.write(record, output, 'fasta')
          print(f'  {fasta} processed.')

def join_multi_gene_FASTAs(file_prefix, gene_list, input_archive_name, delim = '_', user_tag = '', genbank_details = None, folder = True):
  print(f'join_multi_gene_FASTAs: initializing...\n')
  for gene in gene_list: join_single_gene_FASTA(file_prefix = file_prefix,
                                                gene = gene,
                                                input_archive_name = input_archive_name,
                                                delim = delim,
                                                user_tag = user_tag,
                                                genbank_details = genbank_details,
                                                folder = folder)
  concat_dirname = f'{file_prefix}_concat'
  shutil.make_archive(concat_dirname, "zip", concat_dirname)
  print(f'\n  fastas written to {concat_dirname}.zip')

# if the user wants to concatenate a GenBank input with their own files
def genbank_concatenator(file_prefix, input_archive_name,
                         assignment_matrix, genbank_input,
                         user_tag = '_USR'):
  # gets the list of genes from genbank_to_fastas()
  genes, genbank_details = genbank_to_fastas(file_prefix, assignment_matrix, genbank_input)

  # concatenates the user-supplied files
  join_multi_gene_FASTAs(file_prefix = file_prefix,
                         gene_list   = genes,
                         input_archive_name = input_archive_name,
                         user_tag = user_tag,
                         genbank_details = genbank_details)

###### BLAST_wrapper()

In [ ]:
def BLAST_single(BLAST_query, max_hits = 20):
  blastables = ['.fasta', '.fa', '.fas', '.fsa', '.fna', '.ffn', '.faa', '.frn', '.seq']
  if os.path.splitext(BLAST_query)[1].lower() not in blastables:
    print(f'  {BLAST_query} does not appear to be a FASTA file.')
    print(f'  skipping...')
    return
  print(f'  {BLAST_query}: reading...')
  basename  = Path(BLAST_query).stem
  BLAST_out = f'{basename}_BLAST.tsv'

  print(f'  {BLAST_query}: BLASTing...')

  command = f'''./bin/ncbi-blast-2.17.0+/bin/blastn -query  {BLAST_query} -max_target_seqs {max_hits} -out {BLAST_out} -outfmt "7 qseqid sacc stitle length qstart qend pident evalue" -db core_nt -remote'''
  subprocess.run(command, shell=True)

  print(f'  {BLAST_query}: BLASTed.')

def BLAST_multiple(BLAST_dir, max_hits = 20):
  print(f'  attempting to access {BLAST_dir}')
  BLAST_dir = Path(BLAST_dir)
  for filename in os.listdir(BLAST_dir):
    BLAST_single(f'{BLAST_dir}/{filename}', max_hits)

def BLAST_wrapper(input_type, input_name, max_hits = 20):
  if input_type.lower() == 'file':
    BLAST_single(input_name, max_hits)
  elif input_type.lower() in ['dir', 'folder', 'directory']:
    BLAST_multiple(input_name, max_hits)
  else: print("  input_type must be 'file' or 'folder'")

###### MAFFT_wrapper()

In [ ]:
def MAFFT_single(input_fasta, adjustdirection, directory = '.'):
  alignables = ['.fasta', '.fa', '.fas', '.fsa', '.fna', '.ffn', '.faa', '.frn', '.seq']
  if os.path.splitext(input_fasta)[1].lower() not in alignables:
    print(f'\nMAFFT_single:   {input_fasta} does not appear to be a FASTA file. skipping...')
    return

  basename  = Path(input_fasta).stem
  print(f'\nMAFFT_single:   processing {basename}...')
  afa_filename = f'{basename}.fna'

  for dir in ['afa', 'nex', 'logs']:
    os.makedirs(Path(directory) / dir, exist_ok=True)

  afa_folder = Path(directory)  / 'afa'
  afa_out   = Path(afa_folder)  / afa_filename

  nex_folder = Path(directory)  / 'nex'
  nex_out    = Path(nex_folder) / f'{basename}.nex'

  log_folder = Path(directory)  / 'logs'
  MAFFT_log  = Path(log_folder) / f'{basename}.log'


  # builds the MAFFT command
  cmd = ['./bin/mafft-linux64/mafft.bat'] # the base executable
  if adjustdirection == True:
    print(f'\nMAFFT_single:   adjustdirection = True. passing setting to MAFFT.')
    cmd += ['--adjustdirection']
  cmd += ['--auto', str(input_fasta)] # this needs to be at the end

  # sends the assembled command to MAFFT and starts the alignment
  with open(afa_out, 'w') as output:
    with open(MAFFT_log, 'w') as log:
      subprocess.run(cmd, stdout = output, stderr = log, text=True, check=True)
  print(f'                {str(afa_filename)} written successfully.')
  print(f'                MAFFT logs written to {MAFFT_log}.')


  # detects and corrects reverse complement tagging by MAFFT (adds _R_ to IDs)
  # gets the list of IDs in the original FASTA file
  input_records = []
  for record in SeqIO.parse(input_fasta, 'fasta'):
    input_records.append(record.id)

  # parses afa files as alignment objects to return some basic info
  afa_alignment = AlignIO.read(afa_out, 'fasta')
  cols = afa_alignment.get_alignment_length()
  seqs = len(afa_alignment)
  print(f'                {seqs} sequences, {cols} columns (incl. gaps) in {afa_filename}')

  if adjustdirection == True:
    print(f'\nMAFFT_single:   checking for reverse-complement (RC) tags...')
    for record in afa_alignment:
      if(record.id.startswith('_R_') and record.id not in input_records):
        original_record = record.id[3:]
        print(f'                {original_record} RC-tagged by MAFFT. restoring voucher ID...')
        record.id = original_record
        record.description = ''

  # saves the afa, with corrections if applicable
  SeqIO.write(afa_alignment, afa_out, 'fasta')

  # conversion using AlignIO
  with open(nex_out, 'w') as output:
      AlignIO.convert(afa_out, 'fasta',
                      nex_out, 'nexus',
                      molecule_type="DNA")
      print(f'                {afa_filename} converted to Nexus as {basename}.nex.')

# runs MAFFT on each FASTA in the the input folder
def MAFFT_multiple(MAFFT_dir, adjustdirection, directory = '.'):
  print(f'\nMAFFT_multiple: {MAFFT_dir}: attempting to access directory...')
  MAFFT_dir = Path(MAFFT_dir)
  for filename in os.listdir(MAFFT_dir):
    MAFFT_single(MAFFT_dir / filename, adjustdirection, directory = directory)

def nexus_concatenator(nex_folder, prefix = ''):
  # just in case
  if prefix == '': prefix = Path(nex_folder).parent
  supermatrix_filename = f"{prefix}.supermatrix.nex"

  # starts the concatenation process
  print(f'\nnexus_concatenator: examining directory: {nex_folder}...')
  alignables = ['.nexus', '.nex']
  input_nex_filenames = []
  for nexus_file in os.listdir(nex_folder):
    filepath = Path(nex_folder)/nexus_file
    if os.path.splitext(nexus_file)[1].lower() not in alignables:
      print(f'\n                    {nexus_file} does not appear to be a nexus file. skipping...')
    elif filepath.name != supermatrix_filename:
      print(f'\n                    {filepath}: adding to valid nexi...')
      input_nex_filenames.append(filepath)

  # takes the valid nexus files and parses them as alignments
  print(f'\n                    processing nexi...')
  nexi =  [(str(fname), Nexus.Nexus(str(fname))) for fname in input_nex_filenames]
  # gives us some basic info about each alignment
  for nex_filename in input_nex_filenames:
    nex_alignment = AlignIO.read(nex_filename, 'nexus')
    cols = nex_alignment.get_alignment_length()
    seqs = len(nex_alignment)
    print(f'                    {seqs} sequences, {cols} columns (incl. gaps) in {Path(nex_filename).name}')

  # makes the concatenated alignment
  if len(nexi) == 0:
    print(f'                    no valid Nexus files found.')
    return
  else:
    print(f'\nnexus_concatenator: joining nexi...')
    combined_nex = Nexus.combine(nexi)
    concat_filepath = Path(nex_folder) / supermatrix_filename
    with open(concat_filepath, "w") as concat_output:
      combined_nex.write_nexus_data(concat_output)

    # gives basic info about the concatenated alignment
    combined_handle = AlignIO.read(concat_filepath, 'nexus')
    cols = combined_handle.get_alignment_length()
    seqs = len(combined_handle)
    print(f'                    {seqs} sequences, {cols} columns (incl. gaps) in concatenated alignment.')
    print(f'                    concatenated alignment written to:')
    print(f'                    {concat_filepath.resolve()}')

# this function calls MAFFT for each file, converts them to Nexus, and concatenates them
def MAFFT_wrapper(input_type, input_name, prefix = '', concat = True, adjustdirection = True):
  print(f'MAFFT_wrapper:  initializing...')
  if input_type.lower() == 'file':
    MAFFT_single(input_name, adjustdirection)
  elif input_type.lower() in ['dir', 'folder', 'directory']:
    output_foldername = f'{input_name}_aligned'
    os.makedirs(output_foldername, exist_ok=True)
    MAFFT_multiple(input_name, directory = output_foldername, adjustdirection = adjustdirection)
    if concat == True: nexus_concatenator(Path(output_foldername)/ 'nex', prefix = prefix)
    shutil.make_archive(output_foldername, "zip", output_foldername)
    print(f'\nMAFFT_wrapper:  alignments written to {output_foldername}.zip.\n')
  else: print("  input_type must be 'file' or 'folder'")

###### iqtree_wrapper()

In [ ]:
def iqtree_wrapper(input, prefix, bootstraps = 1000, model = 'MFP',
                   nthreads = 'AUTO', redo = False, outgroup = ''):
  iqtree_dir = f'{prefix}_iqtree'
  os.makedirs(iqtree_dir, exist_ok=True)
  exe = str(Path('./bin/iqtree-3.1.3-Linux/bin/iqtree3').resolve())

  if   Path(input).is_dir():  input_flag = '-p' # -p flag for directories
  elif Path(input).is_file(): input_flag = '-s' # -s flag for single files
  cmd = [exe, input_flag, input,
              '-pre', f'{iqtree_dir}/{prefix}',
              '-m', model,
              '-bb', str(bootstraps),
              '-nt', str(nthreads)]

  # if we're not looking for checkpoints, add --redo to overwrite existing runs
  if redo is True: cmd.append('--redo')

  print(f'  attempting to run IQTree with the following parameters:')
  cmd_string = " ".join(cmd)
  print(f'  {cmd_string}')

  with Popen(cmd, stdout=PIPE, bufsize=1, universal_newlines=True) as p:
    for line in p.stdout: print(line, end='') # process line here

  p.wait()
  if p.returncode != 0: raise CalledProcessError(p.returncode, p.args)

  shutil.make_archive(iqtree_dir, "zip", iqtree_dir)
  print(f'\n  IQTree results written to {iqtree_dir}.zip')

  print(f'  IQTree parameters were as follows:')
  print(f'  {cmd_string}')

###### mrbayes_wrapper(), mrbayes_converter()

In [ ]:
def MrBayes_converter(input_nex, prefix):
  MYPAD = ' '*len('MrBayes_converter: ')
  # copies input file to the output folder; MrBayes struggles w/absolute paths
  print(f'MrBayes_converter: preparing to translate sets block for MrBayes...\n')
  nex_basename = Path(input_nex).name
  mb_dir = f'{prefix}_bayes'         # names the output directory

  # reads the input file
  print(f'{MYPAD}locating sets block...\n')
  partition_lines = []
  with open(input_nex, 'r+') as infile:
    copy = False
    setsblock_start = None
    bayesblock_exists = False
    line_counter = 0
    while True:
      seek_position = infile.tell() # builds offset counter
      line = infile.readline()
      if not line: break
      line_counter += 1
      if   'begin mrbayes;' in str(line).lower(): bayesblock_exists = True
      elif 'begin sets;'    in str(line).lower():
        print(f'{MYPAD}sets block found on line {line_counter} (position {seek_position}).\n')
        setsblock_start = seek_position # this is where the set block starts
        copy = True
        newline = 'begin mrbayes;\n'
        partition_lines.append(newline)
      if copy == True:
        if 'end;' in line and copy == True:
          partition_lines.append('end;\n')
          copy = False
        elif 'charset' in line.lower():
         source = Path(line).stem
         newline = re.sub(r"(?<=charset )(.*)(?=\ =)", f'{source}', f'  {line}')
         partition_lines.append(newline)
        elif 'charpartition' in line.lower():
          # gets everything after the '=' and cuts it into genes
          genes = line.split('=')[1].split(',')
          # grabs each gene and strips everything after the ':'
          for i, gene in enumerate(genes): genes[i] = (Path(gene.split(':')[0]).stem).strip()
          newline = f'  partition combined = {len(genes)}: {", ".join(genes)};\n'
          partition_lines.append(newline)
    print(f'{MYPAD}reached end of file.\n')

    if setsblock_start == None:
      if   bayesblock_exists == False:
        print(f'{MYPAD}error: no (sets or mrbayes) block found!\n')
      elif bayesblock_exists == True:
        print(f'{MYPAD}error: sets block already in MrBayes format.\n')
      return
    else:
      # return to set block start position
      print(f'{MYPAD}replacing sets block with the following...\n')
      infile.seek(setsblock_start)
      for newline in partition_lines:
        print(f'                   {newline}', end='')
        infile.write(newline)
      infile.truncate()

def MrBayes_wrapper(prefix, input_nex, partitioned, input_block = None, samplefreq = 1000, runs = 2,
                    generations = 1000000, nchains = None, burnin = None,
                    relburnin = False, burninfrac = None, resume = False):
  MAXPAD  = ' '*(len('MrBayes_converter: '))
  DIFPAD  = ' '*(len('MrBayes_converter: ')-len('MrBayes_wrapper: ')+1)
  # copies input file to the output folder; MrBayes struggles w/absolute paths
  print(f'MrBayes_wrapper:{DIFPAD}creating paths and directories...\n')
  nex_basename = Path(input_nex).name
  mb_dir = f'{prefix}_bayes'         # names the output directory
  os.makedirs(mb_dir, exist_ok=True) # makes our output directory
  shutil.copy(input_nex, mb_dir)
  new_nex = (Path(mb_dir) / nex_basename).resolve()

  print(f'MrBayes_wrapper:{DIFPAD}verifying sets block...\n')
  MrBayes_converter(input_nex = new_nex, prefix = prefix)

  # if we need to build a block from scratch
  if input_block is None:
    block_basename = f'{prefix}_bayesblock.nex'
    blockpath = (Path(mb_dir) / block_basename).resolve()
    print(f'{MAXPAD}creating block file...\n')
    bayesblock = ['begin mrbayes',
                f'  execute {nex_basename}']

    # partitioning info
    if   input_block is None and partitioned == False:
      bayesblock += ['  lset nst=mixed rates=gamma']

    # if the user wants to build a bayes block in NUDIMAX
    elif input_block is None and partitioned == True:
      bayesblock += ['  set partition = combined',
                    '  unlink statefreq=(all) revmat=(all) shape=(all) pinvar=(all), tratio=(all)',
                    '  prset applyto=(all) ratepr=variable',
                    '  lset applyto=(all) rates=gamma']

    # adds mcmc command
    mcmc = f'  mcmc nruns={runs} ngen={generations} samplefreq={samplefreq}'
    if nchains is not None: mcmc += f' nchains={nchains}'
    if resume is True: mcmc += ' append=yes'
    bayesblock += [mcmc]

    # adds optional burnin params
    if burnin is not None:
        if   burnin < 1:
          bayesblock += [f'  sumt relburnin=yes  burninfrac={burnin}',
                         f'  sump relburnin=yes  burninfrac={burnin}']
        elif burnin >= 1:
          bayesblock += [f'  sumt relburnin=no burnin={int(burnin)}',
                         f'  sump relburnin=no burnin={int(burnin)}']
    else: bayesblock +=  ['sumt', 'sump']

      # completes MrBayes formatting
    bayesblock.append('end')
    for i, line in enumerate(bayesblock): bayesblock[i] = f'{line};\n'

    # writes the bayes block file
    with open (blockpath, "w") as output: output.writelines(bayesblock)

  # if the user has a custom/predefined bayes block, we can just use it
  elif input_block is not None:
    shutil.copy(input_block, mb_dir)
    block_basename = (Path(input_block).name)
    blockpath = (Path(mb_dir) / block_basename).resolve()

  # take either our custom-generated block our the user-input block and runs it
  mb_exe = str(Path('./bin/mrbayes-3.2.7/bin/mb').resolve())
  cmd = [mb_exe, block_basename]
  print(f'{MAXPAD}attempting to run MrBayes with the following parameters:')
  print(f'{MAXPAD}{" ".join(cmd)}')

  with Popen(cmd, stdout=PIPE, bufsize=1, cwd = mb_dir, universal_newlines=True) as p:
    # sump and sumt print a LOT of text that's not useful to humans.
    # this mutes it (although it still saves to the output files)
    mute = False
    mute_triggers   = ['General explanation:',
                       'Phylogram (based on average branch lengths):',
                       'Taxon   1 ->']

    unmute_triggers = ['Returning execution to calling file ...',
                       'Clade credibility values:',
                       'Calculating tree probabilities...']
    for line in p.stdout:
      if not mute and any(trigger in line for trigger in mute_triggers):
        print(line, end='')
        print(f'\nMrBayes_wrapper: previous line triggered mute function. pausing output...\n')
        mute = True
      elif mute and any(trigger in line for trigger in unmute_triggers):
        print(line, end='')
        print(f'\nMrBayes_wrapper: previous line triggered unmute function; resuming output...\n')
        mute = False
      elif not mute: print(line, end='') # prints the line

  p.wait()
  if p.returncode != 0:
    raise CalledProcessError(p.returncode, p.args)

  print(f'\n{MAXPAD}MrBayes parameters were as follows:')
  print(f'{MAXPAD}{" ".join(cmd)}\n')

  # prints the bayes block file
  with open(blockpath, "r") as input: print(input.read())

  shutil.make_archive(mb_dir, "zip", mb_dir)
  print(f'MrBayes_wrapper:{DIFPAD}MrBayes results written to:')
  print(f'{MAXPAD}{mb_dir}.zip')

###### leaf_renamer()

In [ ]:
def leaf_renamer(tree_filepath, sample_table, old_names, new_names,
                 sample_suffix = '', trim_parenthesis = False,
                 greedy_trimming = False):
  tree_filename = Path(tree_filepath).name
  tree_parent_dir = Path(tree_filepath).parent
  print(f'leaf_renamer:           reading {tree_filename}...\n')
  tree = Phylo.read(tree_filepath, "newick")

  # makes old and new data. this lifts some code from genbankificator()
  rename_df = pd.read_csv(sample_table)
  print(f'leaf_renamer:           calling sanitize_fasta_headers()...')
  rename_df[old_names] = sanitize_fasta_headers(rename_df[old_names])

  # duplicates the voucher column for renaming
  print(f'leaf_renamer:           removing Newick-breaking characters...\n')
  dedup_names = f'{old_names}_corr'
  rename_df[dedup_names] = rename_df[old_names]
  rename_df[dedup_names] = rename_df[dedup_names].replace("[:,&'’]", '', regex=True)
  # defining the dict to build on

  print(f'leaf_renamer:           checking for duplicate vouchers...\n')
  keycounts = {} # to handle duplicate vouchers (e.g., Kim 2024)
  for index, base_voucher in enumerate(rename_df[old_names]):
      if base_voucher not in keycounts:
       keycounts[base_voucher] = 1
       voucher_new = base_voucher
      elif base_voucher in keycounts:
        keycounts[base_voucher] += 1
        voucher_new = f'{base_voucher}_{keycounts[base_voucher]}'
      rename_df.at[index, dedup_names] = voucher_new

  if   trim_parenthesis ==True and greedy_trimming == False:
    print(f'leaf_renamer:           trimming conservatively...')
    print(f'                        e.g., "Sample A (Gosliner 2026) 02 (Extraction B)" → "Sample_A_02"\n')
    rename_df[new_names] = rename_df[new_names].replace(r"\((.*?)\)", '', regex=True)
  elif trim_parenthesis == True and greedy_trimming == True:
    print(f'leaf_renamer:           trimming greedily...')
    print(f'                        e.g., "Sample_A_(Gosliner 2026) 02 (Extraction B)" → "Sample_A"\n')
    rename_df[new_names] = rename_df[new_names].replace(r"\((.*)\)", '', regex=True)
  elif trim_parenthesis == False and greedy_trimming == True:
    print(f'leaf_renamer:           Note: "greedy_trimming = True" has no effect if "trim_parenthesis = False."\n')
  rename_df[new_names] = rename_df[new_names].replace("[:,&'’().]", '', regex=True)
  rename_df[new_names] = rename_df[new_names].str.strip().replace(' +', '_', regex=True)

  print(f'leaf_renamer:           checking for duplicate vouchers...\n')
  keycounts = {} # to handle duplicate vouchers (e.g., Kim 2024)
  for index, base_voucher in enumerate(rename_df[new_names]):
      if base_voucher not in keycounts:
       keycounts[base_voucher] = 1
       voucher_new = base_voucher
      elif base_voucher in keycounts:
        keycounts[base_voucher] += 1
        voucher_new = f'{base_voucher}_{keycounts[base_voucher]}'
      rename_df.at[index, new_names] = voucher_new

  rename_dict = rename_df.set_index(dedup_names)[new_names].to_dict()

  for leaf in tree.get_terminals():
    # remove the suffix
    leaf.name = leaf.name.replace(sample_suffix, '')
    new_leaf_name = rename_dict.get(leaf.name, 'NODATA')

    if new_leaf_name != 'NODATA':
      leaf.name = new_leaf_name

  output_newick = Path(tree_parent_dir) / f"renamed_{tree_filename}"
  print(f'leaf_renamer:           saving output file to:')
  print(f'                        {output_newick}')
  Phylo.write(tree, output_newick, "newick")

## User-facing commands below
(Click to expand individual functions)

---

**CAUTION: DON'T LOSE YOUR DATA!**

**Google Colab's file storage is cleared every time the runtime is disconnected.**

Files are not saved unless you specifically mount (and save to) your Google Drive.

When working in the Colab temporary storage, remember to regularly download all outputs and log files.

### Google Colab: Export to Google Drive

In [ ]:
# export_to_drive() takes up to three inputs
#   destination   (Optional) is the target file path on Google Drive.
#                 Defaults to NUDIMAX_backup_<timestamp>
#   source        is the file or folder to back up
#   copy_all      (Optional) will copy the entire NUDIMAX scratch folder.

In [ ]:
# @title export_to_drive() {single-column:true}
# @markdown Destination filepath on Google Drive (Default: NUDIMAX_backup_yymmdd-hhmmss)
destination = "" # @param {"type":"string", "placeholder":"Tenellia_backup"}
# @markdown Source
source = '' # @param {"type":"string", "placeholder":"Tenellia_backup"}
# @markdown Copy entire Google Colab scratch directory?
copy_all = False # @param {"type":"boolean"}
export_to_drive(destination = destination,
                source      = '',
                copy_all    = copy_all)

export_to_drive: verifying environment...
export_to_drive: no destination specified. generating default name.
export_to_drive: error: no file specified for backup.
                 select a file to back up, or set copy_all = True.


### Phase 1: Sequence acquisition and intake

#### If you are downloading multi-gene info from GenBank...

In [ ]:
# the input to voucher_matrix_to_genbank() is a CSV spreadsheet that looks like this...
#
# Voucher	      16S	     COI      H3       gene4
# CASIZ 174485  KY128712 KY128917 KY128504 AB123456
# CASIZ 179463a KY128713 KY128918 KY128505 AB789012
# etc...

# genes can be in any order, and additional unique genes can be included

# it is CRITICALLY IMPORTANT that genbank IDs be assigned correctly
#   e.g., the code can't tell if you put a COI entry in the 16S column
#   as of 0.13a, genbank_to_fastas() saves .log files that help identify
#   incorrectly classified accession numbers; it can also (slowly) run BLAST

In [ ]:
# @title voucher_matrix_to_genbank() {single-column:true}
# @markdown Input CSV filename  (see above for example formatting)
genbank_csv_filename = "" # @param {"type":"string","placeholder":"Tenellia_corrected.csv"}
voucher_matrix_to_genbank(genbank_csv_filename);

In [ ]:
# voucher_matrix_to_genbank() has 2 outputs...
#   a _clean.csv file, which has sanitized names to carry forward
#   a _BE.txt file, for upload to Batch Entrez
#      1. go to https://www.ncbi.nlm.nih.gov/sites/batchentrez
#      2. upload the _BE.txt file
#      3. download the BE result as a GenBank file (.gb), rename, and upload
#
# genbank_to_fastas() takes three inputs
#   file_prefix       is a file prefix or project name common to all input files
#   assignment_matrix is the clean CSV file produced by voucher_matrix_to_genbank()
#   genbank_input     is the .gb download from genbank

In [ ]:
# @title genbank_to_fastas() {single-column:true}
# @markdown Desired universal file prefix (i.e., project name)
file_prefix       = "" # @param {"type":"string","placeholder":"Tenellia"}
# @markdown Assignment matrix from voucher_matrix_to_genbank() (ending in _clean.csv)
assignment_matrix = "" # @param {"type":"string","placeholder":"Tenellia_corrected_clean.csv"}
# @markdown Genbank download filename (.gb)
genbank_input     = "" # @param {"type":"string","placeholder":"Tenellia_corrected.gb"}

genbank_to_fastas(file_prefix       = file_prefix,
                  assignment_matrix = assignment_matrix,
                  genbank_input     = genbank_input);

#### Optional detour: batch BLAST


In [ ]:
# BLAST_wrapper() takes three inputs
#   input_type is 'file' or 'folder'
#   input_name is the FASTA or folder name (containing FASTAs)
#   max_hits   is the maximum matches to return per query

# this runs a remote BLAST. it takes a while. go have lunch.

# BLAST_wrapper() creates a .zip archive
#   it contains one TSV file for each FASTA
#   for each sequence in that FASTA, it will display the top n hits
#   this can help identify sequences with incorrect GenBank metadata

In [ ]:
# @title BLAST_wrapper() {single-column:true}
# @markdown Input type (choose either 'folder' or 'file')
input_type = 'folder' # @param ['folder', 'file'] {"type":"string","placeholder":"Tenellia"}
# @markdown Input file or folder name
input_name = '' # @param {"type":"string","placeholder":"Tenellia_out"}
# @markdown Maximum number of hits to display (10 recommended)
max_hits   = None # @param {"type":"integer","placeholder":'10'}

BLAST_wrapper(input_type = input_type,
              input_name = input_name,
              max_hits = max_hits)

#### If you are concatenating input FASTAs for ONE GENE...

In [ ]:
# genbank_to_fasta() creates a .zip archive
#   it contains one folder
#   this contains one FASTA file per gene in the original table input
#   download it, extract it, and add your sequences to the FASTA files
#   this can be done in AliView, BioPython, Geneious, or a text editor

#   it is critical that sequences to be assigned to the same specimen have identical names
#   once sequences are added, compress this folder (.zip) and reupload
#   it is critical that sequences to be assigned to the same specimen have identical names
#   ^ i said that multiple times on purpose, because
#   it is critical that sequences to be assigned to the same specimen have identical names

# join_single_gene_FASTA() takes four arguments:
#   gene               is the gene ID, which must appear at the end of filenames (e.g., 'Tenellia_COI.fasta')
#   file_prefix        will be appended to all files
#   input_archive_name is the name of the .zip archive being uploaded
#   delim              (optional) precedes the gene ID in filenames(default '_')

In [ ]:
# join_single_gene_FASTA() takes four arguments:
#   gene               is the gene ID, which must appear at the end of filenames (e.g., 'Tenellia_COI.fasta')
#   file_prefix        will be appended to all files
#   input_archive_name is the name of the .zip archive being uploaded
#   delim              (optional) precedes the gene ID in filenames(default '_')

# @title join_single_gene_FASTA() {single-column:true}
# @markdown Name of gene (will be appended to filename)
gene               = "" # @param {"type":"string", "placeholder":"COI"}
# @markdown Desired file prefix (i.e., project name)
file_prefix        = "" # @param {"type":"string", "placeholder":"Tenellia"}
# @markdown Input archive name (.zip format)
input_archive_name = "" # @param {"type":"string", "placeholder":"my_Tenellia_fastas.zip"}

join_single_gene_FASTA(gene = gene,
                       file_prefix = file_prefix,
                       input_archive_name = input_archive_name);

#### If you are concatenating FASTAs for MULTIPLE genes...

In [ ]:
# join_multi_gene_FASTAs() takes four arguments:
#   gene_list          is a space-delimited list of gene IDs (e.g., "16S COI H3"; no quotes in Colab)
#   file_prefix        will be appended to all files
#   input_archive_name is the name of the .zip archive being uploaded
#   delim              (optional) precedes the gene ID in filenames(default '_')

In [ ]:
# if you are concatenating FASTAs for MULTIPLE genes...

# join_multi_gene_FASTAs() takes four arguments:
#   gene_list          is a
#   file_prefix        will be appended to all files
#   input_archive_name is the name of the .zip archive being uploaded
#   delim              (optional) precedes the gene ID in filenames(default '_')

# @title join_multi_gene_FASTAs() {single-column:true}
# @markdown Desired universal file prefix (i.e., project name)
file_prefix        ="" # @param {"type":"string", "placeholder":"Tenellia"}
# @markdown Space-delimited list of gene IDs (no quotes)
# THE ABOVE MARKDOWN LINE ONLY APPLIES IF YOU ARE USING THE COLAB FORMS UX.
# IF YOU ARE EDITING THE CODE DIRECTLY, INCLUDE QUOTES AS A NORMAL STRING.
gene_list          = "" # @param {"type":"string", "placeholder":"16S COI H3"}
# @markdown Input archive name (.zip format)
input_archive_name = "" # @param {"type":"string", "placeholder":"my_tenellia_fastas.zip"}

join_multi_gene_FASTAs(file_prefix = file_prefix,
                       gene_list   = gene_list.split(' '),
                       input_archive_name = input_archive_name);

#### If you are concatenating FASTAs for MULTIPLE genes, AND combining it with a GenBank download...


In [ ]:
# genbank_concatenator() takes 4 arguments:
#   file_prefix        is a file prefix (e.g., the project name)
#   assignment_matrix  is the clean CSV file produced by voucher_matrix_to_genbank()
#   genbank_input      is the .gb download from genbank
#   input_archive_name is the name of the .zip archive being uploaded

In [ ]:
# if you are concatenating FASTAs for MULTIPLE genes, AND combining it with a GenBank download...

# genbank_concatenator() takes 4 arguments:
#   file_prefix        is a file prefix (e.g., the project name)
#   assignment_matrix  is the clean CSV file produced by voucher_matrix_to_genbank()
#   genbank_input      is the .gb download from genbank
#   input_archive_name is the name of the .zip archive being uploaded

# @title genbank_concatenator() {single-column:true}
# @markdown Desired universal file prefix (i.e., project name)
file_prefix        = "" # @param {"type":"string", "placeholder":"Tenellia"}
# @markdown Assignment matrix from voucher_matrix_to_genbank() (ending in _clean.csv)
assignment_matrix  = "" # @param {"type":"string", "placeholder":"Tenellia_corrected_clean.csv"}
# @markdown Genbank download filename (.gb)
genbank_input      = "" # @param {"type":"string", "placeholder":"Tenellia_corrected.gb"}
# @markdown Input archive name (.zip format)
input_archive_name = "" # @param {"type":"string", "placeholder":"my_Tenellia_fastas.zip"}

genbank_concatenator(file_prefix        = file_prefix,
                     assignment_matrix  = assignment_matrix,
                     genbank_input      = genbank_input,
                     input_archive_name = input_archive_name);

### Phase 2: Sequence alignment with MAFFT (below; click to expand)
Note: "--adjustdirection" parameter automatically checks for reverse complements

In [ ]:
# MAFFT_wrapper() takes up to five inputs
#   input_type is 'file' or 'folder'
#   input_name is a FASTA filename, or the name of a folder containing FASTAs
#   prefix          (optional, default: input name) is the project name
#   adjustdirection (optional, default: True) MAFFT reverse-complement detection
#   concat          (optional, default: True) creates a concatenated Nexus
#                   supermatrix (.supermatrix.nex) for use in MrBayes

In [ ]:
# MAFFT_wrapper() takes two inputs
#   input_type is 'file' or 'folder'
#   input_name is a FASTA filename, or the name of a folder containing FASTAs

# @title MAFFT_wrapper() {single-column:true}

# @markdown Input type (choose 'folder' or 'file' )
input_type      = "file" # @param ["folder", "file"] {"type":"string"}

# @markdown Input file or folder name
input_name      = "" # @param {"type":"string", "placeholder":"Tenellia_concat"}

# @markdown Project name (optional; if different than input name)
prefix          = "" # @param {"type":"string"}

# @markdown Check for reverse-complemented sequences?
adjustdirection = True # @param {"type":"boolean"}

# @markdown Concatenate alignments into a single NEXUS matrix for MrBayes?
concat          = False # @param {"type":"boolean"}

MAFFT_wrapper(input_type      = input_type,
              input_name      = input_name,
              prefix          = prefix,
              concat          = concat,
              adjustdirection = adjustdirection)

MAFFT_wrapper:  initializing...

MAFFT_single:   processing 282 documents from GenBank Coryphella Fasta...

MAFFT_single:   adjustdirection = True. passing setting to MAFFT.
                282 documents from GenBank Coryphella Fasta.fna written successfully.
                MAFFT logs written to logs/282 documents from GenBank Coryphella Fasta.log.
                282 sequences, 668 columns (incl. gaps) in 282 documents from GenBank Coryphella Fasta.fna

MAFFT_single:   checking for reverse-complement (RC) tags...
                GQ292022.1 RC-tagged by MAFFT. restoring voucher ID...
                HM162694.1 RC-tagged by MAFFT. restoring voucher ID...
                HM162717.1 RC-tagged by MAFFT. restoring voucher ID...
                HM162746.1 RC-tagged by MAFFT. restoring voucher ID...
                HM162749.1 RC-tagged by MAFFT. restoring voucher ID...
                HM162758.1 RC-tagged by MAFFT. restoring voucher ID...
                HQ616748.1 RC-tagged by MAFFT. restor

In [ ]:
# MAFFT_wrapper() creates at least one output:
#   if input_type = 'file', this will be a single aligned FASTA (.afa)
#   if input_type = 'folder', it will be a .ZIP archive containing...
#   - a  "logs" folder, containing MAFFT logs (.log)
#   - an "afa"  folder, containing aligned FASTAs (.afa)
#   - a  "nex"  folder, containing single NEXUS alignments (.nex)
#     if "concat" = True, then there will be a combined file (.supermatrix.nex)

# CAUTION: VISUALLY CHECK YOUR ALIGNMENTS! (e.g., in AliView, Mesquite, etc)
#   the code cannot tell if a sequence is mis-assigned
#     (e.g., if a COI sequence is mixed in with H3 sequences)
#   these errors are typically obvious in alignment viewers and will appear as
#     outliers: one or two very long/short sequences that cause extreme gapping.

#   check the sequences and trim as you see fit
#   BLAST any suspicious-looking sequences to check for incorrect genbank accessions
#   reverse-complementing sequences may be necessary to get a proper alignment
#     (as of v0.13+, this can be handled atomatically by MAFFT)

### Phase 3: Building and cleaning phylogenetic trees (click to expand)

#### IQ-TREE: Maximum-Likelihood (ML) phylogenetic analysis

In [ ]:
# iqtree_wrapper() takes at least two inputs...
#   prefix     is your project name
#   input      is a folder of aligned FASTAs
#   bootstraps (optional) is the number of bootstraps (default: 1000)
#   model      (optional) is the model to use (default: MFP)

In [ ]:
# @title iqtree_wrapper() {single-column:true}
# @markdown Desired output file prefix (i.e., project name)
prefix = '' # @param {"type":"string", "placeholder":"Tenellia"}
# @markdown Input (single alignment or folder containing alignment files)
input = '' # @param {"type":"string", "placeholder":"Tenellia_concat_aligned/afa"}
# @markdown Number of bootstraps (Optional; Default: 1000)
bootstraps = None # @param {"type":"integer", "placeholder":'1000'}
# @markdown Evolutionary model (Optional; Default: MFP)
model = "" # @param {"type":"string", "placeholder":'MFP'}
# @markdown Number of threads (Optional; Default: AUTO)
nthreads = "" # @param {"type":"string", "placeholder":'AUTO'}
# @markdown Overwrite a previously completed run (Optional; Default: False)?
redo      = False # @param {"type":"boolean", "placeholder":"False"}


iqtree_wrapper(prefix = prefix,
               input  = input,
               bootstraps = bootstraps or 1000,
               model = model or 'MFP',
               nthreads = nthreads or 'AUTO',
               redo = redo)

  attempting to run IQTree with the following parameters:
  /content/bin/iqtree-3.1.3-Linux/bin/iqtree3 -s /content/afa/282 documents from GenBank Coryphella Fasta.fna -pre Megan_iqtree/Megan -m MFP -bb 1000 -nt AUTO
IQ-TREE version 3.1.3 for Linux x86 64-bit built Jun 19 2026
Developed by Bui Quang Minh, Thomas Wong, Nhan Ly-Trong, Huaiyan Ren
Contributed by Lam-Tung Nguyen, Dominik Schrempf, Chris Bielow,
Olga Chernomor, Michael Woodhams, Diep Thi Hoang, Heiko Schmidt

Host:    59230b4fddb3 (AVX2, FMA3, 12 GB RAM)
Command: /content/bin/iqtree-3.1.3-Linux/bin/iqtree3_intel -s /content/afa/282 documents from GenBank Coryphella Fasta.fna -pre Megan_iqtree/Megan -m MFP -bb 1000 -nt AUTO
Seed:    173403 (Using SPRNG - Scalable Parallel Random Number Generator)
Time:    Mon Aug 24 22:26:54 2026
Kernel:  AVX+FMA - auto-detect threads (2 CPU cores detected)

Reading alignment file /content/afa/282 documents from GenBank Coryphella Fasta.fna ... Fasta format detected
Reading fasta file: done 

#### MrBayes: Bayesian Inference (BI) phylogenetic analysis

In [ ]:
# MrBayes_wrapper() takes several inputs:
#   prefix      is the desired output file prefix (i.e., project name)
#   input_nex   is the path to the input Nexus alignment file (.nex)
#   runs        (Optional) is the number of MCMC analyses to run; default 2)
#   ncahins     (Optional) is the number of MCMC chains to use; default: 4)
#   burnin      (Optional) is the burn-in time; default 0.25
#               this is an integer (n generations) OR a decimal (0.25)
#   generations (Optional) is the number of generations; default: 1000000)
#   samplefreq  (Optional) is the MCMC sample frequency; default 1000)
#   partitioned (Optional) toggles partitioned analysis on/off; default: False

In [ ]:
# @title MrBayes_wrapper() {single-column:true}
# @markdown Desired output file prefix (i.e., project name)
prefix    = "" # @param {"type":"string", "placeholder":"Tenellia"}
# @markdown Path to input Nexus alignment file (.nex)
input_nex = "" # @param {"type":"string", "placeholder":"./Tenellia_concat_aligned/nex/Tenellia_concat_aligned.supermatrix.nex"}
# @markdown Path to predefined Bayes block file (If you don't have one, leave blank and set the next parameters)
input_block = "" # @param {"type":"string", "placeholder":"./Tenellia_concat_aligned/nex/Tenellia_concat_aligned.bayesblock.nex"}

# @markdown Number of MCMC analyses to run (Optional; default 2)
runs = None    # @param {"type":"integer", "placeholder":"2"}
# @markdown Number of MCMC chains to use (Default: 4)
nchains = None # @param {"type":"integer", "placeholder":"4"}
# @markdown Burn-in time (Optional; number of generations OR a decimal value; Default 0.25)
burnin = "" # @param {"type":"string", "placeholder":"0.25"}
# @markdown Number of generations (Optional; Default: 1000000)
generations = None # @param {"type":"integer", "placeholder":"1000000"}
# @markdown MCMC sample frequency (Optional; default 1000)
samplefreq = None # @param {"type":"integer", "placeholder":"1000"}
# @markdown Is this a partitioned analysis? Default: False
partitioned = True # @param {"type":"boolean", "placeholder":"False"}
# @markdown Attempt to resume a previously aborted run from checkpoint file? Default: False
resume      = True # @param {"type":"boolean", "placeholder":"False"}

MrBayes_wrapper(prefix    = prefix,
                input_nex = input_nex,
                input_block = input_block,
                runs = runs or 2, nchains = nchains or 4,
                burnin = burnin,
                generations = generations or 1000000,
                samplefreq = samplefreq or 1000,
                partitioned = partitioned,
                resume = resume)

#### Optional feature: Rename leaves on your output tree

In [ ]:
# leaf_renamer takes several inputs:
#   tree_filepath is the name of a Newick tree file
#   sample_table  is the name of a  two-column .CSV file
#   old_names     is the name of the column containing the names as they CURRENTLY appear
#   new_names     is the name of the column containing the names as they SHOULD appear
#   sample_suffix    (optional) is any text string to be removed from the end of all samples
#   trim_parenthesis (optional) if True, will trim gently:   A (B) C (D) → A_C
#   greedy_trimming  (optional) if True, will trim greedily: A (B) C (D) → A

In [ ]:
# @title leaf_renamer() {single-column:true}
# @markdown Newick tree filename
tree_filepath = "" # @param {"type":"string", "placeholder":"Tenellia_concat.contree"}
# @markdown Filename of two-column CSV table with old and new filenames
sample_table = "" # @param {"type":"string", "placeholder":"Tenellia_names.csv"}
# @markdown Name of column containing ORIGINAL sample names (e.g., from voucher_matrix_to_genbank())
old_names = "" # @param {"type":"string", "placeholder":"Voucher_old"}
# @markdown Name of column containing DESIRED sample names
new_names = "" # @param {"type":"string", "placeholder":"Voucher_new"}
# @markdown Suffix (any undesired text currently appended to all samples)
sample_suffix = "" # @param {"type":"string", "placeholder":"_Tenellia"}
# @markdown Trim parenthesis (e.g., citation information?)
trim_parenthesis = False # @param {"type":"boolean"}
# @markdown Trim greedily?
greedy_trimming = False # @param {"type":"boolean"}

leaf_renamer(tree_filepath    = tree_filepath,
             sample_table     = sample_table,
             old_names        = old_names,
             new_names        = new_names,
             sample_suffix    = sample_suffix,
             trim_parenthesis = trim_parenthesis,
             greedy_trimming  = greedy_trimming)

leaf_renamer:           reading Megan.contree...

leaf_renamer:           calling sanitize_fasta_headers()...
sanitize_fasta_headers: removing MAFFT/IQ-TREE-breaking characters...

leaf_renamer:           removing Newick-breaking characters...

leaf_renamer:           checking for duplicate vouchers...

leaf_renamer:           checking for duplicate vouchers...

leaf_renamer:           saving output file to:
                        /content/Megan_iqtree/renamed_Megan.contree
